In [1]:
import chromadb

In [2]:
#1.创建客户端
client=chromadb.Client()

In [3]:
#2.获取或创建
collection1=client.get_or_create_collection('collection1')
collection2=client.get_or_create_collection('collection2')

## 新增数据(C)

In [4]:
collection1.add(
    ids=['4','5','6'],
    documents=[
        "我闻琵琶已叹息",
        "凄凄惨惨戚戚",
        "断肠人在天涯",
    ],
    metadatas=[
        {'描述':'诗句'},
        {'描述':'好听的诗句'},
        {'描述':'夕阳西下'}
    ]
)

In [5]:
collection1.peek()

{'ids': ['4', '5', '6'],
 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
         -0.01157282, -0.12148048],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [6]:
collection1.peek()['embeddings'].shape

(3, 384)

In [10]:
#2.添加数据的同时,指定嵌入向量
collection2.add(
    ids=['4','5','6'],
    documents=[
        "我闻琵琶已叹息",
        "凄凄惨惨戚戚",
        "断肠人在天涯",
    ],
    embeddings=[
        [1,2,3],
        [4,5,6],
        [7,8,9]
    ]
)

In [13]:
print(collection2.peek())
collection2.peek()['embeddings'].shape

{'ids': ['4', '5', '6'], 'embeddings': array([[1., 2., 3.],
       [4., 5., 6.],
       [7., 8., 9.]]), 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [None, None, None]}


(3, 3)

In [14]:
collection2.add(
    ids='7',
    documents='我花开后百花杀'
)
#InvalidArgumentError: Collection expecting embedding with dimension of 3, got 384;
# 维度不同,因为前面插入3维向量,必须相同的向量维度,也要指定向量维度为三维向量


InvalidArgumentError: Collection expecting embedding with dimension of 3, got 384

In [26]:
collection2.add(
    ids='7',
    documents='我花开后百花杀',
    embeddings=[2,1,9]
)

In [28]:
collection2.add(
    ids='8',
    # documents='我花开后百花杀',
    embeddings=[21,11,91]
)
#因为只插入了id和向量,document默认为None

In [27]:
collection1.add(
    ids='7',
    documents='我花开后百花杀',
    embeddings=[2,1,9]
)
#InvalidArgumentError: Collection expecting embedding with dimension of 384, got 3
#同理,因为collection1是默认的编码器生成的向量,是384维的向量,所以如果指定向量维度,会插入错误

InvalidArgumentError: Collection expecting embedding with dimension of 384, got 3

In [29]:
collection2.peek()

{'ids': ['4', '5', '6', '7', '8'],
 'embeddings': array([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [ 2.,  1.,  9.],
        [21., 11., 91.]]),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯', '我花开后百花杀', None],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None, None, None, None, None]}

## 查询数据

In [46]:
#1.query方法
# client.list_collections()
print(collection1.peek())
# print(collection2.peek())
#通过文本直接查询
print(collection1.query(
    query_texts='白居易',
    n_results=10
))
#通过where对元数据过滤来查询
print(collection1.query(
    query_texts='白居易',
    where={
        '描述':{'$eq':'诗句'}
    }
))
# 通过where_document对documents进行查询
print('4',collection1.query(
    query_texts='琵琶行',
    where_document={
        '$contains':'天涯'
    }
))

collection1.peek()

{'ids': ['4', '5', '6'], 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
        -0.01157282, -0.12148048],
       [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
        -0.03452881, -0.00614677],
       [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
        -0.07604755,  0.02073891]], shape=(3, 384)), 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}
{'ids': [['6', '4', '5']], 'embeddings': None, 'documents': [['断肠人在天涯', '我闻琵琶已叹息', '凄凄惨惨戚戚']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'描述': '夕阳西下'}, {'描述': '诗句'}, {'描述': '好听的诗句'}]], 'distances': [[0.9272914528846741, 1.0235599279403687, 1.196514368057251]]}
{'ids': [['4']], 'embeddings': None, 'documents': [['我闻琵琶已叹息']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data

{'ids': ['4', '5', '6'],
 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
         -0.01157282, -0.12148048],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [47]:
#通过向量来查询
print('1111                  ',collection2.query(
    query_embeddings=[1,2,3]
))

1111                   {'ids': [['4', '5', '7', '6', '8']], 'embeddings': None, 'documents': [['我闻琵琶已叹息', '凄凄惨惨戚戚', '我花开后百花杀', '断肠人在天涯', None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None, None, None]], 'distances': [[0.0, 27.0, 38.0, 108.0, 8225.0]]}


In [16]:
#1.直接获取数据
collection1.get(ids='4')

{'ids': ['4'],
 'embeddings': None,
 'documents': ['我闻琵琶已叹息'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'描述': '诗句'}]}

In [24]:
collection1.get(ids=["4","5","6"],include=["documents", "embeddings", "metadatas"])

{'ids': ['4', '5', '6'],
 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
         -0.01157282, -0.12148048],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['documents', 'embeddings', 'metadatas'],
 'data': None,
 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [25]:
#2.增加筛选条件查询
collection1.get(
    where={'描述':{'$eq':'诗句'}},
    where_document={'$contains':'我'}
)

{'ids': ['4'],
 'embeddings': None,
 'documents': ['我闻琵琶已叹息'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'描述': '诗句'}]}

## 更新数据


In [49]:
collection1.peek()

{'ids': ['4', '5', '6'],
 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
         -0.01157282, -0.12148048],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [50]:
collection1.add(
    ids='4',
    documents='乍暖还寒时候',
    metadatas={'作者':'李清照'}
                )

In [51]:
collection1.peek()

{'ids': ['4', '5', '6'],
 'embeddings': array([[-0.01978007,  0.0973409 ,  0.02793549, ...,  0.06545459,
         -0.01157282, -0.12148048],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [52]:
collection1.update(
    ids='4',
    documents='乍暖还寒时候',
    metadatas={'作者':'李清照'}
                )

In [54]:
collection1.peek()

{'ids': ['4', '5', '6'],
 'embeddings': array([[ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.02066   ,  0.12249423,  0.07895101, ...,  0.06120238,
         -0.07604755,  0.02073891]], shape=(3, 384)),
 'documents': ['乍暖还寒时候', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句', '作者': '李清照'}, {'描述': '好听的诗句'}, {'描述': '夕阳西下'}]}

In [55]:
collection1.upsert(
    ids='6',
    documents='北国风光,千里冰封,万里雪飘',
    metadatas={'作者':'毛泽东'}
                )
collection1.upsert(
    ids='1',
    documents='五洲震荡和为贵',
    metadatas={'作者':'匿名'}
                )
collection1.peek()

{'ids': ['4', '5', '6', '1'],
 'embeddings': array([[ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.00495984,  0.0755002 ,  0.06164318, ..., -0.00944335,
         -0.01271691,  0.00043947],
        [ 0.01210416,  0.1096666 ,  0.01525423, ...,  0.05245707,
          0.03042525,  0.03459564]], shape=(4, 384)),
 'documents': ['乍暖还寒时候', '凄凄惨惨戚戚', '北国风光,千里冰封,万里雪飘', '五洲震荡和为贵'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'作者': '李清照', '描述': '诗句'},
  {'描述': '好听的诗句'},
  {'作者': '毛泽东', '描述': '夕阳西下'},
  {'作者': '匿名'}]}

## 删除数据

In [56]:
collection1.peek()

{'ids': ['4', '5', '6', '1'],
 'embeddings': array([[ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.00495984,  0.0755002 ,  0.06164318, ..., -0.00944335,
         -0.01271691,  0.00043947],
        [ 0.01210416,  0.1096666 ,  0.01525423, ...,  0.05245707,
          0.03042525,  0.03459564]], shape=(4, 384)),
 'documents': ['乍暖还寒时候', '凄凄惨惨戚戚', '北国风光,千里冰封,万里雪飘', '五洲震荡和为贵'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'描述': '诗句', '作者': '李清照'},
  {'描述': '好听的诗句'},
  {'描述': '夕阳西下', '作者': '毛泽东'},
  {'作者': '匿名'}]}

In [57]:
collection1.delete(ids='5')

In [58]:
collection1.peek()

{'ids': ['4', '6', '1'],
 'embeddings': array([[ 0.06183338,  0.06178976,  0.03821023, ..., -0.00555789,
         -0.03452881, -0.00614677],
        [ 0.00495984,  0.0755002 ,  0.06164318, ..., -0.00944335,
         -0.01271691,  0.00043947],
        [ 0.01210416,  0.1096666 ,  0.01525423, ...,  0.05245707,
          0.03042525,  0.03459564]], shape=(3, 384)),
 'documents': ['乍暖还寒时候', '北国风光,千里冰封,万里雪飘', '五洲震荡和为贵'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'作者': '李清照', '描述': '诗句'},
  {'描述': '夕阳西下', '作者': '毛泽东'},
  {'作者': '匿名'}]}